# 📚 Interactive Lexical Retrieval (BM25 / FTS5)

Run **lexical retrieval** over the Stage-6 chunk store with an input query.

This notebook is **CPU-only and dependency-light** — it only needs the Python
standard library (`sqlite3`) plus `pandas`/`pyarrow` (light deps). The heavy
GPU / dense-retrieval packages (`torch`, `transformers`, `FlagEmbedding`,
`faiss`, `neo4j`, `accelerate`, `sentence-transformers`) were uninstalled from
this machine; this lexical baseline runs without any of them.

**How to use**
1. Run the *Setup* cell to open the FTS5 index.
2. Edit the `QUERY` string in the *Run a query* cell (or pick one from the
   dev-set examples) and run it.
3. Inspect the ranked hits + full chunk text below.

## 1. Setup — open the Stage-6 FTS5 index

In [ ]:
import sys, time
from pathlib import Path

# Make the project's `retrieval` package importable.
ROOT = Path.cwd().resolve()
# If running from notebooks/, step up to the project root.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from retrieval.bm25_index import FTSIndex, tokenize_query
from retrieval.retriever import make_relevant_lists

DB = ROOT / "data" / "stage6_data" / "chunk_store.sqlite"
print("DB:", DB)
print("exists:", DB.exists())

# 'bm25_ranked'  -> global BM25 ranking (matches the PLAN baseline; slower).
# 'fts_fast'     -> fast MATCH + LIMIT, no global sort (faster, weaker recall).
FTS_MODE = "bm25_ranked"

fts = FTSIndex(str(DB), mode=FTS_MODE).open()
print(f"rows={fts.n_rows:,}  lexical_backend={fts.lexical_backend!r}  mode={FTS_MODE!r}")

## 2. Run a query

Edit `QUERY` below (Vietnamese legal questions work best) and run the cell.

In [ ]:
# ──────────────────────────  EDIT YOUR QUERY HERE  ──────────────────────────
QUERY = "Thủ tục đăng ký doanh nghiệp lần đầu bao gồm những bước nào?"
# ────────────────────────────────────────────────────────────────────────────

TOP_K = 10          # number of hits to retrieve from FTS5
SHOW_TEXT = True    # fetch + show full chunk_text for each hit
TEXT_PREVIEW = 600  # chars of chunk_text to display per hit (0 = full text)

print(f"Tokens: {tokenize_query(QUERY)}")
t0 = time.time()
hits = fts.search(QUERY, top_k=TOP_K)
print(f"Retrieved {len(hits)} hits in {time.time()-t0:.3f}s\n")

if not hits:
    print("No hits — try a different query or switch FTS_MODE to 'fts_fast'.")
else:
    if SHOW_TEXT:
        full = {r["row_idx"]: r for r in fts.fetch_chunks([h["row_idx"] for h in hits])}
    for i, h in enumerate(hits, 1):
        score = h.get("bm25_score", "")
        score_str = f"  bm25={score:+.4f}" if isinstance(score, float) else ""
        print(f"#{i}  row_idx={h['row_idx']}{score_str}")
        print(f"    law_id    : {h['law_id']}")
        print(f"    ten_van_ban: {h['ten_van_ban']}")
        print(f"    dieu_so   : {h['dieu_so']}")
        if SHOW_TEXT:
            txt = full.get(h["row_idx"], {}).get("chunk_text", "")
            if TEXT_PREVIEW and len(txt) > TEXT_PREVIEW:
                txt = txt[:TEXT_PREVIEW] + " …[truncated]"
            print(f"    chunk_text:\n{txt}")
        print("-" * 100)

    # Collapse to submission-style relevant_docs / relevant_articles.
    # Wrap dicts so make_relevant_lists (which expects Hit-like attributes) works.
    from types import SimpleNamespace
    ns = [SimpleNamespace(**h, chunk_text=full.get(h["row_idx"], {}).get("chunk_text", "") if SHOW_TEXT else "") for h in hits]
    docs, articles = make_relevant_lists(ns)
    print("\nrelevant_docs:", docs)
    print("relevant_articles:", articles)

## 3. Dev-set example questions (optional)

Convenience picker — set `PICK` to a 1-based index and run to load that
question into `QUERY`, then re-run the Section 2 cell.

In [ ]:
import json

QUESTIONS = json.loads((ROOT / "dev_set" / "questions.json").read_text(encoding="utf-8"))
for q in QUESTIONS:
    print(f"{q['id']:>2}. {q['question']}")

In [ ]:
PICK = 1   # 1..20
QUERY = QUESTIONS[PICK - 1]["question"]
print("QUERY set to:")
print(QUERY)

## 4. Cleanup

In [ ]:
fts.close()
print("FTS index closed.")